## Prerequisites
1. Create (or reuse) a Kaggle dataset that contains the Sputnik-SR repository **and** the `data/sputnik.db` SQLite file. Attach it to this notebook as an input dataset.
2. Enable the **GPU (T4 x2)** accelerator and make sure internet access is allowed (to install specific TensorFlow wheels).
3. Adjust the hyperparameters in the training command cell if you need different epochs, batch sizes, or evaluation limits.

In [ ]:
# Copy the repository + DB from the attached Kaggle dataset into /kaggle/working
import shutil
from pathlib import Path


INPUT_DATASET = Path("/kaggle/input/sputnik-sr/")
CANDIDATES = sorted(p for p in INPUT_DATASET.iterdir() if (p / "data" / "sputnik.db").exists())
if not CANDIDATES:
    raise FileNotFoundError(
        "Attach a dataset that exposes data/sputnik.db (e.g., /kaggle/input/sputnik-sr/...)."
    )

SOURCE_DIR = CANDIDATES[0]
REPO_DIR = Path("/kaggle/working/sputnik-SR")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
shutil.copytree(SOURCE_DIR, REPO_DIR)
print("Repository ready at", REPO_DIR)
print("Database located at", REPO_DIR / "data" / "sputnik.db")
# Optional: show git metadata if present
git_info = REPO_DIR / ".git" / "HEAD"
if git_info.exists():
    print("Git HEAD:", git_info.read_text().strip())

In [ ]:
# Install TensorFlow (GPU) and any extra dependencies
!pip install -q --upgrade pip
!pip install -q 'tensorflow==2.15.0' 'tensorflow-io-gcs-filesystem==0.34.0'

In [ ]:
# Verify GPU availability and enable memory growth
import tensorflow as tf


gpus = tf.config.list_physical_devices("GPU")
if not gpus:
    raise SystemError("No GPU detected. Enable GPU in Kaggle notebook settings.")
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception as exc:  # noqa: BLE001
        print("Unable to set memory growth on", gpu, exc)
print("Visible GPUs:", gpus)
print("TensorFlow version:", tf.__version__)

In [ ]:
# Compose the Two Towers training command (edit hyperparameters as needed)
import textwrap
from pathlib import Path


REPO_DIR = Path("/kaggle/working/sputnik-SR")
PYTHON = "python"  # Kaggle runtime default
BUILD_SCRIPT = REPO_DIR / "offline_recommender" / "build_two_towers.py"
DATABASE = REPO_DIR / "data" / "sputnik.db"
MODELS_DIR = REPO_DIR / "models" / "Two Towers"
CHECKPOINT_PATH = MODELS_DIR / "checkpoints" / "two_towers_kaggle.weights.h5"
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)

TRAIN_CMD = textwrap.dedent(
    f"""
    PYTHONPATH={REPO_DIR} {PYTHON} {BUILD_SCRIPT} \
      --database {DATABASE} \
      --embedding-dim 64 \
      --epochs 20 \
      --batch-size 2048 \
      --learning-rate 0.001 \
      --min-user-ratings 5 \
      --min-release-ratings 3 \
      --num-negatives 4 \
      --max-genres 10 \
      --evaluate-ndcg \
      --ndcg-holdout 0.2 \
      --ndcg-k 9 \
      --ndcg-max-users 500 \
      --checkpoint-path {CHECKPOINT_PATH} \
      --resume-from-checkpoint {CHECKPOINT_PATH} \
      --verbose
    """
).strip()
print(TRAIN_CMD)

In [ ]:
# Launch training (this can take several hours on the full dataset)
import subprocess


print("Starting Two Towers training...")
result = subprocess.run(TRAIN_CMD, shell=True, check=False)
print("Return code:", result.returncode)
if result.returncode != 0:
    raise RuntimeError("Training command failed; check the logs above for details.")

In [ ]:
# Archive artifacts so they appear under the Kaggle "Output" section for download
import shutil
import zipfile
from pathlib import Path


MODELS_DIR = Path("/kaggle/working/sputnik-SR/models/Two Towers")
OUTPUT_ZIP = Path("/kaggle/working/two_towers_artifacts.zip")
if OUTPUT_ZIP.exists():
    OUTPUT_ZIP.unlink()
with zipfile.ZipFile(OUTPUT_ZIP, "w", zipfile.ZIP_DEFLATED) as zipf:
    for path in MODELS_DIR.rglob("*"):
        zipf.write(path, path.relative_to(MODELS_DIR.parent))
print("Artifacts zipped at", OUTPUT_ZIP)